# BA · 05 Dashboard Ejecutivo


## 🎯 Objetivos

Al final de este notebook:
- [ ] Calcularás 5 KPIs críticos de supply chain
- [ ] Crearás visualizaciones ejecutivas con Plotly
- [ ] Entenderás cómo interpretar cada métrica
- [ ] Construirás un dashboard básico interactivo

## 📊 Contexto de Supply Chain

**Problema de negocio**: Los líderes necesitan visibilidad rápida del desempeño global de la cadena sin navegar múltiples reportes.

**Impacto esperado**: Decisiones más rápidas, identificación temprana de problemas, comunicación efectiva con stakeholders.

**Cuando usar**: Reuniones ejecutivas semanales, revisiones mensuales de desempeño, presentaciones a directorio.

## 📦 Configuración del Entorno

In [1]:
# Importar librerías estándar
import sys
from pathlib import Path
import warnings
from datetime import datetime, timedelta

# Datos y análisis
import numpy as np
import pandas as pd

# Visualización
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px

# Reproducibilidad
np.random.seed(42)
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)
warnings.filterwarnings('ignore')

# Detectar raíz del repo
repo_root = Path.cwd()
while repo_root != repo_root.parent:
    if (repo_root / 'pyproject.toml').exists():
        break
    repo_root = repo_root.parent

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print(f"✅ Entorno listo")
print(f"   Python: {sys.version.split()[0]}")
print(f"   pandas: {pd.__version__}")
print(f"   Raíz: {repo_root}")

✅ Entorno listo
   Python: 3.12.10
   pandas: 2.3.3
   Raíz: f:\GitHub\supply-chain-data-notebooks


## 📥 Carga de Datos

In [2]:
# Ruta a datos
data_dir = repo_root / 'data' / 'raw'

# Cargar datasets necesarios
orders = pd.read_csv(data_dir / 'orders.csv', parse_dates=['date'])
inventory = pd.read_csv(data_dir / 'inventory.csv')
transport = pd.read_csv(data_dir / 'transport_events.csv')
locations = pd.read_csv(data_dir / 'locations.csv')

print(f"📊 Datasets cargados:")
print(f"   Orders: {len(orders):,} registros")
print(f"   Inventory: {len(inventory):,} registros")
print(f"   Transport: {len(transport):,} registros")
print(f"   Locations: {len(locations):,} ubicaciones")

📊 Datasets cargados:
   Orders: 8,504 registros
   Inventory: 3,000 registros
   Transport: 2,995 registros
   Locations: 30 ubicaciones


## 🔢 KPI 1: OTIF (On-Time In-Full)

**Definición**: % de órdenes entregadas completas y a tiempo.

**Meta típica**: > 95%

**Impacto**: Satisfacción del cliente, reducción de reclamos.

In [3]:
# Simular datos de entrega (en dataset real vendrían del sistema)
orders['delivered_qty'] = orders['qty']
orders['delivered_on_time'] = np.random.choice([True, False], len(orders), p=[0.92, 0.08])
orders['delivered_complete'] = orders['delivered_qty'] == orders['qty']

# Calcular OTIF
orders['otif'] = orders['delivered_on_time'] & orders['delivered_complete']
otif_rate = orders['otif'].mean() * 100

print(f"📊 OTIF Global: {otif_rate:.1f}%")

# OTIF por canal
otif_by_channel = orders.groupby('channel')['otif'].mean() * 100

fig = go.Figure()
fig.add_trace(go.Bar(
    x=otif_by_channel.index,
    y=otif_by_channel.values,
    marker_color=['green' if x >= 95 else 'orange' if x >= 90 else 'red' for x in otif_by_channel.values],
    text=[f"{v:.1f}%" for v in otif_by_channel.values],
    textposition='outside'
))
fig.add_hline(y=95, line_dash="dash", line_color="green", annotation_text="Meta: 95%")
fig.update_layout(
    title="📦 OTIF por Canal de Venta",
    xaxis_title="Canal",
    yaxis_title="OTIF (%)",
    height=400
)
fig.show()

📊 OTIF Global: 92.0%


## 📈 KPI 2: Fill Rate (Tasa de Servicio)

**Definición**: % de la demanda servida sin quiebres de stock.

**Meta típica**: > 98%

**Impacto**: Ventas perdidas, erosión de marca.

In [4]:
# Simular disponibilidad (en real: match orders vs inventory disponible)
orders['served_qty'] = orders['qty'] * np.random.uniform(0.95, 1.0, len(orders))
fill_rate = (orders['served_qty'].sum() / orders['qty'].sum()) * 100

print(f"📊 Fill Rate Global: {fill_rate:.1f}%")

# Fill rate por categoría (join con products)
# Simplificado para demo
fill_by_month = orders.set_index('date').resample('M')['qty'].count()

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=fill_by_month.index,
    y=np.random.uniform(96, 99.5, len(fill_by_month)),
    mode='lines+markers',
    name='Fill Rate',
    line=dict(color='blue', width=3),
    fill='tozeroy'
))
fig.add_hline(y=98, line_dash="dash", line_color="green", annotation_text="Meta: 98%")
fig.update_layout(
    title="📊 Fill Rate Mensual",
    xaxis_title="Mes",
    yaxis_title="Fill Rate (%)",
    height=400
)
fig.show()

📊 Fill Rate Global: 97.5%


## 🔄 KPI 3: Inventory Turns (Rotación de Inventario)

**Definición**: Cuántas veces se vende el inventario promedio en un período.

**Meta típica**: Depende de industria (retail: 5-8, farma: 12-15)

**Impacto**: Capital de trabajo, obsolescencia, costos de almacenaje.

In [5]:
# Calcular COGS (Cost of Goods Sold) aproximado
orders['cogs'] = orders['qty'] * 50  # Asumir costo promedio
total_cogs = orders['cogs'].sum()

# Inventario promedio
avg_inventory_value = inventory['on_hand'].sum() * 50

# Inventory turns (anualizado)
days_in_period = (orders['date'].max() - orders['date'].min()).days
turns = (total_cogs / avg_inventory_value) * (365 / days_in_period)

print(f"📊 Inventory Turns (anual): {turns:.1f}x")
print(f"   Days of Inventory: {365/turns:.0f} días")

# Crear gauge chart
fig = go.Figure(go.Indicator(
    mode="gauge+number+delta",
    value=turns,
    title={'text': "🔄 Inventory Turns (Anual)"},
    delta={'reference': 6, 'suffix': 'x'},
    gauge={
        'axis': {'range': [None, 12]},
        'bar': {'color': "darkblue"},
        'steps': [
            {'range': [0, 4], 'color': "lightgray"},
            {'range': [4, 8], 'color': "lightblue"},
            {'range': [8, 12], 'color': "lightgreen"}
        ],
        'threshold': {
            'line': {'color': "green", 'width': 4},
            'thickness': 0.75,
            'value': 6
        }
    }
))
fig.update_layout(height=400)
fig.show()

📊 Inventory Turns (anual): 1.9x
   Days of Inventory: 197 días


## 💰 KPI 4: Costo Logístico (% de Ventas)

**Definición**: Total de costos logísticos como % de ventas netas.

**Meta típica**: 5-8% (varía por industria)

**Impacto**: Rentabilidad, competitividad, pricing.

In [6]:
# Simular costos logísticos
total_sales = orders['qty'].sum() * 100  # Precio promedio venta
logistics_cost = {
    'Transporte': 0.025 * total_sales,
    'Almacenaje': 0.015 * total_sales,
    'Manejo': 0.010 * total_sales,
    'Otros': 0.008 * total_sales
}
total_logistics = sum(logistics_cost.values())
logistics_pct = (total_logistics / total_sales) * 100

print(f"📊 Costo Logístico: {logistics_pct:.1f}% de ventas")

# Desglose en waterfall
fig = go.Figure(go.Waterfall(
    x=list(logistics_cost.keys()) + ['Total'],
    y=[v/total_sales*100 for v in logistics_cost.values()] + [logistics_pct],
    measure=['relative']*len(logistics_cost) + ['total'],
    text=[f"{v:.1f}%" for v in list(logistics_cost.values()) + [logistics_pct]],
    textposition='outside',
    connector={'line': {'color': 'rgb(63, 63, 63)'}}
))
fig.update_layout(
    title="💰 Desglose de Costo Logístico (% ventas)",
    yaxis_title="% de Ventas",
    height=400
)
fig.show()

📊 Costo Logístico: 5.8% de ventas


## ⚡ KPI 5: Perfect Order Rate

**Definición**: % de órdenes perfectas (a tiempo, completa, sin daños, documentación correcta).

**Meta típica**: > 90%

**Impacto**: Costos ocultos (devoluciones, reenvíos, créditos).

In [7]:
# Perfect order = on time + complete + no damage + correct docs
orders['no_damage'] = np.random.choice([True, False], len(orders), p=[0.97, 0.03])
orders['correct_docs'] = np.random.choice([True, False], len(orders), p=[0.96, 0.04])
orders['perfect'] = orders['otif'] & orders['no_damage'] & orders['correct_docs']

perfect_rate = orders['perfect'].mean() * 100
print(f"📊 Perfect Order Rate: {perfect_rate:.1f}%")

# Pareto de problemas
issues = {
    'Retraso': (~orders['delivered_on_time']).sum(),
    'Incompleto': (~orders['delivered_complete']).sum(),
    'Daño': (~orders['no_damage']).sum(),
    'Documentación': (~orders['correct_docs']).sum()
}
issues_df = pd.DataFrame(list(issues.items()), columns=['Tipo', 'Cantidad'])
issues_df = issues_df.sort_values('Cantidad', ascending=False)
issues_df['Acumulado%'] = (issues_df['Cantidad'].cumsum() / issues_df['Cantidad'].sum() * 100)

fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(
    go.Bar(x=issues_df['Tipo'], y=issues_df['Cantidad'], name='Cantidad', marker_color='indianred'),
    secondary_y=False
)
fig.add_trace(
    go.Scatter(x=issues_df['Tipo'], y=issues_df['Acumulado%'], name='% Acumulado', 
               marker_color='blue', mode='lines+markers'),
    secondary_y=True
)
fig.update_layout(title="📊 Pareto de Problemas en Órdenes", height=400)
fig.update_yaxes(title_text="Cantidad de Órdenes", secondary_y=False)
fig.update_yaxes(title_text="% Acumulado", secondary_y=True)
fig.show()

📊 Perfect Order Rate: 85.4%


## 📊 Dashboard Consolidado

Integración de todos los KPIs en un solo vistazo.

In [8]:
# Crear dashboard con subplot
from plotly.subplots import make_subplots

fig = make_subplots(
    rows=2, cols=3,
    subplot_titles=('OTIF', 'Fill Rate', 'Inventory Turns', 
                    'Costo Logístico', 'Perfect Order', 'Tendencia Órdenes'),
    specs=[[{'type': 'indicator'}, {'type': 'indicator'}, {'type': 'indicator'}],
           [{'type': 'indicator'}, {'type': 'indicator'}, {'type': 'scatter'}]],
    vertical_spacing=0.15
)

# OTIF
fig.add_trace(go.Indicator(
    mode="number+delta",
    value=otif_rate,
    title={'text': "OTIF (%)"},
    delta={'reference': 95, 'relative': False},
    number={'suffix': '%'}
), row=1, col=1)

# Fill Rate
fig.add_trace(go.Indicator(
    mode="number+delta",
    value=fill_rate,
    title={'text': "Fill Rate (%)"},
    delta={'reference': 98, 'relative': False},
    number={'suffix': '%'}
), row=1, col=2)

# Inventory Turns
fig.add_trace(go.Indicator(
    mode="number+delta",
    value=turns,
    title={'text': "Inventory Turns"},
    delta={'reference': 6, 'relative': False},
    number={'suffix': 'x'}
), row=1, col=3)

# Costo Logístico
fig.add_trace(go.Indicator(
    mode="number+delta",
    value=logistics_pct,
    title={'text': "Costo Log. (% ventas)"},
    delta={'reference': 6, 'relative': False},
    number={'suffix': '%'}
), row=2, col=1)

# Perfect Order
fig.add_trace(go.Indicator(
    mode="number+delta",
    value=perfect_rate,
    title={'text': "Perfect Order (%)"},
    delta={'reference': 90, 'relative': False},
    number={'suffix': '%'}
), row=2, col=2)

# Tendencia órdenes diarias
daily_orders = orders.set_index('date').resample('D').size()
fig.add_trace(go.Scatter(
    x=daily_orders.index,
    y=daily_orders.values,
    mode='lines',
    name='Órdenes/día',
    line=dict(color='blue', width=2),
    fill='tozeroy'
), row=2, col=3)

fig.update_layout(
    title_text="📊 Dashboard Ejecutivo de Supply Chain",
    height=600,
    showlegend=False
)
fig.show()

print("\n✅ Dashboard generado exitosamente")


✅ Dashboard generado exitosamente


## 📈 Conclusiones y Recomendaciones

### Hallazgos principales:
1. **OTIF:** {otif_rate:.1f}% - {'✅ Dentro de meta' if otif_rate >= 95 else '⚠️ Requiere atención'}
2. **Fill Rate:** {fill_rate:.1f}% - {'✅ Saludable' if fill_rate >= 98 else '⚠️ Revisar disponibilidad'}
3. **Inventory Turns:** {turns:.1f}x - Días de inventario: {365/turns:.0f}
4. **Costo Logístico:** {logistics_pct:.1f}% - {'✅ Competitivo' if logistics_pct <= 6 else '⚠️ Oportunidad de optimización'}
5. **Perfect Order:** {perfect_rate:.1f}% - Enfocarse en principales problemas del Pareto

### Recomendaciones para la operación:
- **Acción 1**: Si OTIF < 95%, revisar cumplimiento de promesas y capacidad de transporte
- **Acción 2**: Si Inventory Turns < 5x, evaluar SKUs de baja rotación para descontinuar
- **Acción 3**: Implementar revisión semanal de estos 5 KPIs con equipos operativos

### Próximos pasos:
- [ ] Automatizar este dashboard con datos en tiempo real
- [ ] Agregar alertas automáticas cuando KPIs salgan de meta
- [ ] Desagregar por región/producto/cliente para análisis drill-down
- [ ] Implementar benchmarking contra competencia/industria

## 🔧 Útiles / Funciones Reutilizables

In [9]:
def calculate_otif(orders_df, date_col='date', qty_col='qty'):
    """
    Calculate OTIF (On-Time In-Full) rate.
    
    Args:
        orders_df: DataFrame with orders
        date_col: Column name for date
        qty_col: Column name for quantity
        
    Returns:
        float: OTIF rate as percentage
    """
    if 'otif' not in orders_df.columns:
        raise ValueError("DataFrame must have 'otif' column")
    return orders_df['otif'].mean() * 100


def create_kpi_indicator(value, title, reference, suffix='%'):
    """
    Create a Plotly indicator for KPI display.
    
    Args:
        value: Current KPI value
        title: KPI name
        reference: Target/reference value
        suffix: Unit suffix
        
    Returns:
        go.Indicator: Plotly indicator object
    """
    return go.Indicator(
        mode="number+delta",
        value=value,
        title={'text': title},
        delta={'reference': reference, 'relative': False},
        number={'suffix': suffix}
    )


print("✅ Funciones auxiliares definidas")

✅ Funciones auxiliares definidas


<div style="width: 100%; clear: both; margin: 0 0 20px 0; border-top: 1px solid #eaecef; padding-top: 24px;"><div style="display: flex; justify-content: space-between; align-items: center; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Helvetica, Arial, sans-serif;"><div style="flex: 1; text-align: left;"><a href="BA-04-supplier_performance.ipynb" style="text-decoration: none; color: #0366d6; font-size: 14px; font-weight: 600; transition: color 0.2s;">← Anterior: [BA-04-supplier_performance.ipynb](../40_business_analytics_bi/BA-04-supplier_performance.ipynb)</a></div><div style="flex: 1; text-align: center; font-size: 14px;"><a href="../../README.md" style="color: #0366d6; text-decoration: none; font-weight: 600; margin: 0 10px;">📑 Índice</a><span style="color: #6a737d;">|</span><a href="../../config/notebooks_index.yml" style="color: #0366d6; text-decoration: none; font-weight: 600; margin: 0 10px;">📋 Catálogo</a></div><div style="flex: 1; text-align: right;"><span style="color: #6a737d; font-size: 14px; cursor: default;">Siguiente →</span></div></div></div>